In [3]:
import os
import yfinance as yf
import time
import math
import smtplib
from email.message import EmailMessage
import pandas as pd 

C = {'G': '\033[92m', 'R': '\033[91m', 'Y': '\033[93m', 'B': '\033[94m', 'C': '\033[96m', 'W': '\033[0m'}

ATR_MULTIPLIER = 2.0 
TRAILING_STOP_PCT = 0.015     
TP_MULTIPLIER = 3.0           
PARTIAL_TP_MULTIPLIER = 1.5   
COOLDOWN_MINUTES = 15         

PORTFOLIO_CAPITAL = 100000.0  
RISK_PER_TRADE_PCT = 0.01     

entry_price = 0               
position_type = 'NONE'        
last_signal = 'NONE'
highest_price = 0             
lowest_price = 999999         
current_shares = 0  
take_profit_price = 0         
breakeven_active = False      
partial_profit_taken = False  
last_exit_time = 0            

total_pnl = 0.0
wins, losses = 0, 0

def log_system_event(message):
    timestamp = time.ctime()
    with open('system_log.txt', 'a') as f:
        f.write(f'[{timestamp}], {message}\n')
    
def send_trade_notification(signal, price, pnl, shares):
    try:
        msg = EmailMessage()
        msg.set_content(f'Bot Alert: {signal}\nPrice: {price:.2f}\nShares: {shares}\nPnL: ₹{pnl:.2f}')
        msg['Subject'] = f'Trade Alert: {signal}'
        
        SENDER_EMAIL = 'smtp.quant@gmail.com'
        APP_PASSWORD = 'My Passcode'
        
        msg['From'] = 'smtp.quant@gmail.com'
        msg['To'] = 'smtp.quant@gmail.com'
        
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login('smtp.quant@gmail.com', 'My Passcode')
            smtp.send_message(msg)
    except Exception as e:
        print(f"\n{C['R']} Email Notification Failed: {e}{C['W']}")

print(f"{C['B']} Initializing MTF Multi-Timeframe Engine...{C['W']}")
master_df = yf.download('RELIANCE.NS', period='7d', interval='1m', progress=False)
print(f"{C['G']} System Online: Multiple Timeframe Trend Filters Engaged{C['W']}\n")

while True:
    try:
        new_data = yf.download('RELIANCE.NS', period='1d', interval='1m', progress=False)
        if not new_data.empty:
            master_df = pd.concat([master_df, new_data.tail(1)])
            master_df = master_df[~master_df.index.duplicated(keep='last')].tail(900) # Kept large window for safety
        else:
            time.sleep(10); continue
        master_df['SMA_45'] = master_df['Close']['RELIANCE.NS'].rolling(window=45).mean()
        master_df['SMA_190'] = master_df['Close']['RELIANCE.NS'].rolling(window=190).mean()
        master_df['SMA_750'] = master_df['Close']['RELIANCE.NS'].rolling(window=750).mean()
        
        delta = master_df['Close']['RELIANCE.NS'].diff()
        master_df['RSI'] = 100 - (100 / (1 + (delta.clip(lower=0).rolling(14).mean() / delta.clip(upper=0).abs().rolling(14).mean())))
        
        master_df['EMA_12'] = master_df['Close']['RELIANCE.NS'].ewm(span=12, adjust=False).mean()
        master_df['EMA_26'] = master_df['Close']['RELIANCE.NS'].ewm(span=26, adjust=False).mean()
        master_df['MACD'] = master_df['EMA_12'] - master_df['EMA_26']
        master_df['MACD_Signal'] = master_df['MACD'].ewm(span=9, adjust=False).mean()

        high, low, pc = master_df['High']['RELIANCE.NS'], master_df['Low']['RELIANCE.NS'], master_df['Close']['RELIANCE.NS'].shift(1)
        tr = pd.concat([high-low, (high-pc).abs(), (low-pc).abs()], axis=1).max(axis=1)
        master_df['ATR'] = tr.rolling(window=14).mean()
        
        latest = master_df.iloc[-1]
        current_price = latest['Close']['RELIANCE.NS'].item()
        current_rsi = latest['RSI'].item()
        sma_45, sma_190 = latest['SMA_45'].item(), latest['SMA_190'].item()
        macro_baseline = latest['SMA_750'].item() # Day 38 Baseline Extract
        current_macd, current_macd_signal = latest['MACD'].item(), latest['MACD_Signal'].item()
        current_atr = latest['ATR'].item()

        # --- RISK MANAGEMENT (DYNAMIC DIRECTIONAL BOUNDARIES) ---
        if position_type in ['LONG', 'SHORT'] and current_shares > 0:
            
            stop_distance = current_atr * ATR_MULTIPLIER
            
            if position_type == 'LONG':
                dynamic_stop_loss = entry_price - stop_distance
                if current_price >= (entry_price + current_atr) and not breakeven_active:
                    breakeven_active = True
                    print(f"\n{C['C']} LONG BREAKEVEN: Stop-Loss moved to entry (₹{entry_price:.2f}){C['W']}")
                if breakeven_active: dynamic_stop_loss = entry_price
                    
            elif position_type == 'SHORT':
                dynamic_stop_loss = entry_price + stop_distance
                if current_price <= (entry_price - current_atr) and not breakeven_active:
                    breakeven_active = True
                    print(f"\n{C['C']} SHORT BREAKEVEN: Stop-Loss moved to entry (₹{entry_price:.2f}){C['W']}")
                if breakeven_active: dynamic_stop_loss = entry_price

            # Partial Take-Profit Scaling
            if not partial_profit_taken:
                if position_type == 'LONG' and current_price >= (entry_price + (current_atr * PARTIAL_TP_MULTIPLIER)):
                    trigger_partial = True
                elif position_type == 'SHORT' and current_price <= (entry_price - (current_atr * PARTIAL_TP_MULTIPLIER)):
                    trigger_partial = True
                else:
                    trigger_partial = False
                    
                if trigger_partial:
                    partial_profit_taken = True
                    shares_to_sell = math.floor(current_shares / 2)
                    if shares_to_sell > 0:
                        pnl = (current_price - entry_price) * shares_to_sell if position_type == 'LONG' else (entry_price - current_price) * shares_to_sell
                        total_pnl += pnl
                        PORTFOLIO_CAPITAL += pnl
                        current_shares -= shares_to_sell
                        print(f"\n{C['G']}💸 PARTIAL TP ({position_type})! Banked: ₹{pnl:.2f}. Residual size running risk-free!{C['W']}")
                        send_trade_notification(f'PARTIAL_TP_{position_type}', current_price, pnl, shares_to_sell)

            if position_type == 'LONG':
                if current_price > highest_price: highest_price = current_price
                trailing_trigger = (current_price - highest_price) / highest_price <= -TRAILING_STOP_PCT
            elif position_type == 'SHORT':
                if current_price < lowest_price: lowest_price = current_price
                trailing_trigger = (lowest_price - current_price) / lowest_price <= -TRAILING_STOP_PCT

            trigger = None
            if position_type == 'LONG':
                if current_price >= take_profit_price: trigger = 'TAKE_PROFIT'
                elif current_price <= dynamic_stop_loss: trigger = 'BREAKEVEN_STOP' if breakeven_active else 'ATR_STOP_LOSS'
                elif trailing_trigger: trigger = 'TRAILING_STOP'
            elif position_type == 'SHORT':
                if current_price <= take_profit_price: trigger = 'TAKE_PROFIT'
                elif current_price >= dynamic_stop_loss: trigger = 'BREAKEVEN_STOP' if breakeven_active else 'ATR_STOP_LOSS'
                elif trailing_trigger: trigger = 'TRAILING_STOP'
            
            if trigger:
                pnl = (current_price - entry_price) * current_shares if position_type == 'LONG' else (entry_price - current_price) * current_shares
                total_pnl += pnl
                PORTFOLIO_CAPITAL += pnl 
                if pnl > 0: wins += 1 
                else: losses += 1
                
                color = C['G'] if pnl >= 0 else C['R']
                print(f"\n{color} {position_type} {trigger} HIT! Cleared {current_shares} shares at {current_price:.2f} | PnL: ₹{pnl:.2f}{C['W']}")
                
                with open('trade_log.csv', 'a') as f:
                    f.write(f'{time.ctime()},{current_price},{trigger}_{position_type},{pnl}\n')
                send_trade_notification(f"{trigger}_{position_type}", current_price, pnl, current_shares)
                
                last_exit_time = time.time()
                position_type = 'NONE'
                entry_price, highest_price, current_shares, take_profit_price = 0, 0, 0, 0
                lowest_price = 999999
                breakeven_active = False 
                partial_profit_taken = False
                last_signal = 'NONE'
                time.sleep(60); continue
        macro_bullish = current_price > macro_baseline if not pd.isna(macro_baseline) else False
        macro_bearish = current_price < macro_baseline if not pd.isna(macro_baseline) else False
        
        if (sma_45 > sma_190) and (current_rsi < 60) and (current_macd > current_macd_signal) and macro_bullish:
            current_signal = 'LONG'
        elif (sma_45 < sma_190) and (current_rsi > 40) and (current_macd < current_macd_signal) and macro_bearish:
            current_signal = 'SHORT'
        else:
            current_signal = 'NONE'

        if current_signal != last_signal and current_shares == 0:
            if current_signal in ['LONG', 'SHORT']:
                minutes_since_exit = (time.time() - last_exit_time) / 60
                
                if minutes_since_exit >= COOLDOWN_MINUTES:
                    entry_price = current_price
                    position_type = current_signal
                    
                    risk_amount = PORTFOLIO_CAPITAL * RISK_PER_TRADE_PCT
                    stop_distance = current_atr * ATR_MULTIPLIER if current_atr > 0 else 0.5
                    current_shares = math.floor(risk_amount / stop_distance) 
                    
                    if position_type == 'LONG':
                        dynamic_stop_loss = entry_price - stop_distance
                        take_profit_price = entry_price + (current_atr * TP_MULTIPLIER)
                        highest_price = entry_price
                    elif position_type == 'SHORT':
                        dynamic_stop_loss = entry_price + stop_distance
                        take_profit_price = entry_price - (current_atr * TP_MULTIPLIER)
                        lowest_price = entry_price
                        
                    breakeven_active = False
                    partial_profit_taken = False
                    
                    print(f"\n{C['G']} ENTRY ALERT: {position_type} at {entry_price:.2f}{C['W']}")
                    print(f"{C['C']}   ↳ Stop-Loss: ₹{dynamic_stop_loss:.2f} | Target: ₹{take_profit_price:.2f}{C['W']}")
                    
                    with open('trade_log.csv', 'a') as f:
                        f.write(f'{time.ctime()},{current_price},{position_type},0\n')
                    send_trade_notification(position_type, current_price, 0, current_shares)
                    
                    last_signal = current_signal
            else:
                last_signal = current_signal

        if position_type == 'LONG': pnl_val = (current_price - entry_price) * current_shares
        elif position_type == 'SHORT': pnl_val = (entry_price - current_price) * current_shares
        else: pnl_val = 0
            
        p_col = C['G'] if pnl_val >= 0 else C['R']
        mins_passed = (time.time() - last_exit_time) / 60

        if pd.isna(macro_baseline): macro_txt = "WARMING_UP"
        elif current_price > macro_baseline: macro_txt = f"{C['G']}BULLISH TIDE{C['W']}"
        else: macro_txt = f"{C['R']}BEARISH TIDE{C['W']}"
        
        if position_type != 'NONE':
            status_display = f"{C['C']}{position_type} Target: ₹{take_profit_price:.2f}{C['W']}"
        elif mins_passed < COOLDOWN_MINUTES:
            status_display = f"{C['Y']} Lockout: {COOLDOWN_MINUTES - mins_passed:.1f}m left{C['W']}"
        else:
            status_display = f" Scanning | Macro: {macro_txt}"
        
        print(f"\r{C['B']}[{time.strftime('%H:%M:%S')}] {C['W']}"
              f"Prc: {current_price:.2f} | "
              f"{status_display} | "
              f"Trade: {p_col}₹{pnl_val:+.2f}{C['W']} | "
              f"Acct: ₹{PORTFOLIO_CAPITAL:.2f} | "
              f"W/L: {wins}/{losses}    ", end="")

    except Exception as e:
        print(f"\n{C['R']} CRITICAL ERROR: {e}{C['W']}"); time.sleep(15)
    time.sleep(60)

 Initializing MTF Multi-Timeframe Engine...
 System Online: Multiple Timeframe Trend Filters Engaged


 ENTRY ALERT: SHORT at 1275.30
   ↳ Stop-Loss: ₹1276.47 | Target: ₹1273.54
[12:43:09] Prc: 1275.50 | SHORT Target: ₹1273.54 | Trade: ₹-170.56 | Acct: ₹100000.00 | W/L: 0/0    
 SHORT ATR_STOP_LOSS HIT! Cleared 853 shares at 1276.10 | PnL: ₹-682.34
[13:01:37] Prc: 1276.50 |  Scanning | Macro: BEARISH TIDE | Trade: ₹+0.00 | Acct: ₹99317.66 | W/L: 0/1    
 ENTRY ALERT: SHORT at 1276.10
   ↳ Stop-Loss: ₹1276.90 | Target: ₹1274.90
[13:03:45] Prc: 1275.80 | SHORT Target: ₹1274.90 | Trade: ₹+372.21 | Acct: ₹99317.66 | W/L: 0/1    
 SHORT BREAKEVEN: Stop-Loss moved to entry (₹1276.10)
[13:05:47] Prc: 1275.60 | SHORT Target: ₹1274.90 | Trade: ₹+620.50 | Acct: ₹99317.66 | W/L: 0/1    
💸 PARTIAL TP (SHORT)! Banked: ₹805.95. Residual size running risk-free!

 SHORT TAKE_PROFIT HIT! Cleared 621 shares at 1274.80 | PnL: ₹807.25
[14:06:41] Prc: 1272.00 |  Scanning | Macro: BEARISH TIDE | Trade: ₹+0.

KeyboardInterrupt: 